In [ ]:
BASE_NAME = 'Noriana López Dataset'
INPUT_FILE_NAME = f'processed_data/西语/{BASE_NAME}.xlsx'
SHEET_FILE_NAME_STAGE_1 = f'train_dataset/西语/{BASE_NAME} Stage1.csv'
SHEET_FILE_NAME_STAGE_2 = f'train_dataset/西语/{BASE_NAME} Stage2 (Gemini).jsonl'
SHEET_FILE_NAME_STAGE_3 = f'train_dataset/西语/{BASE_NAME} Stage3 (OSS).jsonl'

In [ ]:
import pandas as pd

In [ ]:
data = pd.read_excel(INPUT_FILE_NAME)
data

In [ ]:
data = data.to_dict(orient='records')
len_data = len(data)
print(f'Len: {len(data)}')
data[0].keys()

In [ ]:
import json

def save_jsonl(data_list: list, file_path: str):
    with open(file_path, 'w', encoding='utf-8') as f:
        for item in data_list:
            json.dump(item, f, ensure_ascii=False)
            f.write('\n')

# 构建 SFT 数据

In [ ]:
from threading import RLock
from vortezwohl.concurrent import ThreadPool

threads = ThreadPool(128)
train_dataset = []
train_dataset_lock = RLock()


def context_chunk_analyse(origin: str, translation: str, polish: str):
    if len(train_dataset) > len_data:
        return
    with train_dataset_lock:
        train_dataset.append({
            'origin': origin,
            'translation': translation,
            'polish': polish
        })
        if len(train_dataset) % 25 == 0:
            pd.DataFrame(train_dataset).to_csv(SHEET_FILE_NAME_STAGE_1, index=False)


for d_item in data:
    threads.submit(context_chunk_analyse, origin=d_item['章节内容'], translation=d_item['机翻内容'],
                   polish=d_item['翻译精修'])


res = threads.wait_all()
res[0].traceback, res[0].returns

In [ ]:
pd.DataFrame(train_dataset).to_csv(SHEET_FILE_NAME_STAGE_1, index=False)

In [ ]:
import pandas as pd

data = pd.read_csv(SHEET_FILE_NAME_STAGE_1).to_dict(orient='records')
data[0].keys()

In [ ]:
from prompt4py import GeneralTemplate

def format_prompt(content: str) -> str:
    prompt_template = GeneralTemplate()
    prompt_template.role = '你是精通**拉美西班牙语**的文学翻译家.'
    prompt_template.objective = '将给定内容片段(见[Content])准确且自然地翻译为**拉美西班牙语**'
    prompt_template.input = {
        'Content': '{{content}}'
    }
    return prompt_template.render(content=content, markdown=True)

In [ ]:
dataset = []

for d_item in data:
    dataset.append({
        'contents': [
            {
                'role': 'user',
                'parts': [{'text': format_prompt(content=d_item['origin'])}]
            },
            {
                'role': 'model',
                'parts': [{'text': d_item['polish']}]
            }
        ]
    })

save_jsonl(dataset, SHEET_FILE_NAME_STAGE_2)

In [ ]:
oss_dataset = []

for d_item in data:
    oss_dataset.append({'prompt': format_prompt(content=d_item['origin']), 'completion': d_item['polish']})

save_jsonl(oss_dataset, SHEET_FILE_NAME_STAGE_3)